# GLAAM Address Matching Notebook

Matches LDC (London Datastore) messy addresses against OS NGD reference data
to find UPRN links.

## Strategy

Neither the LDC nor the OS data has perfect, consistently-structured address
components. To maximise recall we run **six matching rounds**, each representing
a different way of encoding addresses into a single string:

| Round | Canonical key (OS)                         | Messy key (LDC)              |
|-------|--------------------------------------------|------------------------------|
| **a** | org name + street + postcode               | org name + street + postcode |
| **b** | fields 1-4 + street + postcode              | full address tokens + postcode |
| **c** | building name + number + street + sector   | ↑ same messy b               |
| **d** | building name + subname + street + sector  | ↑ same messy b               |
| **e** | number + street + sector                   | ↑ same messy b               |
| **f** | name + subname + number + street + sector  | ↑ same messy b               |

For each round, `AddressMatcher` runs an exact-match pass followed by a
Splink probabilistic pass. Results are unioned and **hit rules** are applied
to decide which matches to accept.

## Hit rules (post-matching)

A premises is considered **matched** if any of these conditions hold:
1. `fuzz_similarity == 100`  — the input and reference address strings are identical
2. `fuzz_similarity > 95` **and** `method == 'a'` — org-name method, near-exact
3. `distinguishability IS NOT NULL` — Splink found a uniquely distinguishing token

This notebook produced **~77% coverage** on the March 2026 LDC dataset.

## Prerequisites

Data files required (place in `../data/` relative to this notebook):
- `../data/ngd/` — OS NGD ZIP files (downloaded from the run-mar-2026 folder)
- `../data/ldc/ldc_history_YYYY-MM-DD.csv` — set `LDC_FILE` below

In [45]:
# ============================================================
#  CONFIGURATION — edit these paths before running
# ============================================================
import os

# Folder that holds your data (ngd/ and ldc/ subfolders)
DATA_DIR     = os.path.join("data")

NGD_DATA_DIR = os.path.join(DATA_DIR, "ngd")                          # OS NGD CSV(s)
LDC_FILE     = os.path.join(DATA_DIR, "ldc", "ldc_history_2025-12-31.csv")

# Pre-computed results from a previous run (colleague's output or your own).
# If this file exists you can skip Sections 3-8 and jump straight to Section 9.
RESULTS_CSV  = os.path.join("results.csv")

## 1. Imports and DuckDB setup

In [46]:
import multiprocessing
import os
import tempfile
import time
import zipfile

import duckdb
import numpy as np
import pandas as pd
import psutil
from thefuzz import fuzz

from uk_address_matcher import (
    AddressMatcher,
    ExactMatchStage,
    SplinkStage,
)
from uk_address_matcher.linking_model.matching.stages import PeeledAddressStage, UniqueTrigramStage

# -----------------------------------------------------------------
# DuckDB connection — configured once, never replaced
# -----------------------------------------------------------------
con = duckdb.connect(database=":memory:")

cores = multiprocessing.cpu_count()
con.execute(f"PRAGMA threads={cores}")

# Respect DUCKDB_MEMORY_LIMIT env var (set by Docker); else use 90% RAM
container_memory = os.environ.get("DUCKDB_MEMORY_LIMIT", "")
if container_memory:
    con.execute(f"PRAGMA memory_limit='{container_memory}'")
    mem_label = container_memory
else:
    total_gb  = psutil.virtual_memory().total // (1024 ** 3)
    mem_limit = int(total_gb * 0.9)
    con.execute(f"PRAGMA memory_limit='{mem_limit}GB'")
    mem_label = f"{mem_limit}GB (auto)"

temp_dir = tempfile.gettempdir()
con.execute(f"SET temp_directory='{temp_dir}'")

print(f"DuckDB ready — threads: {cores}, memory: {mem_label}, temp: {temp_dir}")

DuckDB ready — threads: 8, memory: 9GB (auto), temp: /tmp


## 2. Helper functions

In [47]:
# Column sets expected in OS NGD product files
_DEFAULT_COLS = [
    "uprn", "organisationname", "subname", "name", "number",
    "streetname", "locality", "townname",
    "primaryclassificationdescription", "postcode",
    "fulladdress", "latitude", "longitude",
]
_ALT_COLS = [
    "uprn", "alternatesubname", "alternatename", "alternatenumber",
    "streetname", "locality", "townname", "postcode",
    "fulladdress", "addressstatus",
]
_PSTL_COLS = [
    "uprn", "organisationname", "subbuildingname", "buildingname",
    "buildingnumber", "thoroughfare", "dependentlocality",
    "posttown", "postcode",
]


def _number_str(series: pd.Series) -> pd.Series:
    """Convert nullable integer series to strings, replacing <NA> with empty."""
    return series.astype(str).replace("<NA>", "").replace("nan", "")


def _load_ngd_csv(path: str) -> pd.DataFrame:
    """Load one OS NGD CSV and normalise column names based on product type."""
    fname = os.path.basename(path)
    if "_altadd" in fname:
        df = pd.read_csv(path, usecols=_ALT_COLS)
        df = df.rename(columns={
            "alternatesubname": "subname",
            "alternatename":    "name",
            "alternatenumber":  "number",
            "addressstatus":    "source",
        })
        df["source"] = "alternative"
    elif "_pstladd" in fname:
        df = pd.read_csv(path, usecols=_PSTL_COLS)
        df = df.rename(columns={
            "subbuildingname":   "subname",
            "buildingname":      "name",
            "buildingnumber":    "number",
            "thoroughfare":      "streetname",
            "dependentlocality": "locality",
            "posttown":          "townname",
        })
        df["source"] = "postal"
        df["number"] = pd.array(df["number"], dtype="Int64")
    else:
        df = pd.read_csv(path, usecols=_DEFAULT_COLS)
        df["source"] = "main"
    df["number_str"] = _number_str(df["number"])
    df["type"] = fname.split("_")[2].split(".")[0]
    return df


def _join(*parts) -> str:
    """Join non-NaN address parts with ', '."""
    return ", ".join(
        str(p).upper() for p in parts if pd.notna(p) and str(p) not in ("", "nan", "NAN")
    )


def build_canonical_key(
    df: pd.DataFrame,
    fields: list[str],
    uid_prefix: str = "c",
) -> pd.DataFrame:
    """
    Build a single-column canonical address key DataFrame from the OS dataframe.

    Parameters
    ----------
    df      : OS NGD dataframe (output of `load_os_df`)
    fields  : column names to concatenate, in order
    uid_prefix : prefix for the uid column name

    Returns
    -------
    DataFrame with columns [f'uid_{uid_prefix}', 'uprn', 'address_c']
    """
    subset = df[["uprn"] + fields].copy()
    subset["address_c"] = [
        _join(*row) for row in subset[fields].itertuples(index=False)
    ]
    result = subset[["uprn", "address_c"]].drop_duplicates().sort_values("uprn")
    result = result.reset_index(drop=True).reset_index().rename(
        columns={"index": f"uid_{uid_prefix}"}
    )
    result[f"uid_{uid_prefix}"] = result[f"uid_{uid_prefix}"].astype(str)
    return result


def build_messy_key(
    df: pd.DataFrame,
    fields: list[str],
    uid_prefix: str = "m",
) -> pd.DataFrame:
    """
    Build a single-column messy address key DataFrame from the LDC dataframe.

    Parameters
    ----------
    df      : LDC dataframe (output of `load_ldc_df`)
    fields  : column names to concatenate, in order
    uid_prefix : prefix for the uid column name

    Returns
    -------
    DataFrame with columns [f'uid_{uid_prefix}', 'uprn', 'premises_id', 'address_m']
    """
    subset = df[["uprn", "premises_id"] + fields].copy()
    subset["address_m"] = [
        _join(*row) for row in subset[fields].itertuples(index=False)
    ]
    result = subset[["uprn", "premises_id", "address_m"]].drop_duplicates().sort_values("premises_id")
    result = result.reset_index(drop=True).reset_index().rename(
        columns={"index": f"uid_{uid_prefix}"}
    )
    result[f"uid_{uid_prefix}"] = result[f"uid_{uid_prefix}"].astype(str)
    return result


def fuzzy_similarity(row) -> int:
    """fuzz.ratio between address_m and address_c columns."""
    s1 = str(row["address_m"]) if pd.notna(row["address_m"]) else ""
    s2 = str(row["address_c"]) if pd.notna(row["address_c"]) else ""
    return fuzz.ratio(s1, s2)


print("Helper functions defined.")

Helper functions defined.


## 2b. Load pre-computed results (shortcut)

If you already have a `results.csv` from a previous run (or from a colleague),
run this cell to load `full_results` directly and then **skip to Section 9**.

The file must have columns: `uid_m`, `premises_id`, `address_m`, `uprn_c`,
`address_c`, `match_weight`, `fuzz_similarity`, `distinguishability`, `method`.

If you want to re-run the full pipeline from raw data, skip this cell and
continue from Section 3.

In [48]:
if os.path.exists(RESULTS_CSV):
    full_results = pd.read_csv(RESULTS_CSV, dtype={"premises_id": str, "uprn_c": str, "uid_m": str})
    # Normalise column names in case the file used different conventions
    full_results = full_results.rename(columns={
        "uid_m":    "uid_m",
        "uprn_c":   "uprn_c",
        "address_m": "address_m",
        "address_c": "address_c",
    })
    # Recompute fuzz_similarity if column is all zeros / missing
    if full_results["fuzz_similarity"].fillna(0).eq(0).all():
        print("Recomputing fuzz_similarity...")
        full_results["fuzz_similarity"] = full_results.apply(fuzzy_similarity, axis=1)
    print(f"Loaded {len(full_results):,} rows from {RESULTS_CSV}")
    print(f"Columns: {list(full_results.columns)}")
    print(f"Unique premises : {full_results['premises_id'].nunique():,}")
    print()
    print("full_results is ready.  Jump to Section 9 (hit rules) or Section 11 (labelling).")
    full_results.head(3)
else:
    print(f"No pre-computed results found at '{RESULTS_CSV}'.")
    print("Continue from Section 3 to run the full matching pipeline.")

Loaded 1,089,058 rows from results.csv
Columns: ['uid_m', 'uprn_m', 'premises_id', 'address_m', 'uid_c', 'match_weight', 'distinguishability', 'fuzz_similarity', 'uprn_c', 'address_c', 'method']
Unique premises : 127,161

full_results is ready.  Jump to Section 9 (hit rules) or Section 11 (labelling).


## 3. Load and clean OS NGD reference data

The OS NGD data may arrive either as:
- A **single large CSV** (e.g. `add_gb_builtaddress.csv`) — the full GB built-address file
- **ZIP files** each containing multiple product CSVs (`_altadd`, `_pstladd`, main)

Both formats are supported.  The cells below:
1. Extract any ZIPs not already extracted
2. Load all CSVs into a single pandas DataFrame
3. Forward-fill classification and coordinates across address variants
4. Drop residential addresses (we only match commercial premises)
5. Add `postcode_sector` = postcode minus the last 2 characters

> **Note:** If you have the large single-file CSV the ZIP step is a no-op.

In [49]:
# Extract ZIPs (skips files already extracted)
zip_list = [
    f for f in os.listdir(NGD_DATA_DIR)
    if f.endswith(".zip") and "streetaddress" not in f
]
for zf in zip_list:
    zip_path = os.path.join(NGD_DATA_DIR, zf)
    with zipfile.ZipFile(zip_path) as z:
        to_extract = [
            m for m in z.namelist()
            if "rltenty.csv" not in m and "otrclass.csv" not in m
        ]
        for member in to_extract:
            dest = os.path.join(NGD_DATA_DIR, member)
            if not os.path.exists(dest):
                z.extract(member, NGD_DATA_DIR)

print(f"ZIPs processed: {len(zip_list)}")

ZIPs processed: 0


In [50]:
# Load all CSVs
csv_files = [
    os.path.join(NGD_DATA_DIR, f)
    for f in os.listdir(NGD_DATA_DIR)
    if f.endswith(".csv")
]
print(f"Loading {len(csv_files)} CSV files...")

os_df = pd.concat([_load_ngd_csv(f) for f in csv_files], axis=0)
os_df = os_df.replace(pd.NA, np.nan)
print(f"Raw rows: {len(os_df):,}")

Loading 12 CSV files...
Raw rows: 11,208,627


In [51]:
# Forward-fill classification and coordinates across address variants,
# then drop residentials
os_df = os_df.sort_values(["uprn", "primaryclassificationdescription"])
os_df["primaryclassificationdescription"] = (
    os_df.groupby("uprn", sort=False)["primaryclassificationdescription"].ffill()
)
os_df = os_df[os_df["primaryclassificationdescription"] != "Residential"]
os_df["latitude"]  = os_df.groupby("uprn", sort=False)["latitude"].ffill()
os_df["longitude"] = os_df.groupby("uprn", sort=False)["longitude"].ffill()

# Deduplicate on key fields
os_df = os_df.drop_duplicates(
    subset=["uprn", "organisationname", "subname", "name",
            "number_str", "streetname", "townname", "postcode"],
    keep="first",
)

# Postcode sector = postcode without the last 2 characters  e.g. 'SW1W 0LN' -> 'SW1W 0'
os_df["postcode_sector"] = os_df["postcode"].str[:-2]

# Strip apostrophes and periods from text fields (matches colleague's preprocessing)
for _col in ["organisationname", "subname", "name", "streetname"]:
    os_df[_col] = os_df[_col].str.replace("'", "", regex=False).str.replace(".", "", regex=False)

print(f"Commercial rows after cleaning: {len(os_df):,}")
os_df[["uprn", "organisationname", "name", "number", "streetname", "postcode"]].head(3)

Commercial rows after cleaning: 1,454,910


,uprn,organisationname,name,number,streetname,postcode
307076,5000025,NaN,LAURISTON LODGE,NaN,BARLOW ROAD,NW6 2BH
226572,5000025,NaN,LAURISTON LODGE,NaN,BARLOW ROAD,NW6 2BH
307077,5000033,TOOK TOOK,NaN,221,WEST END LANE,NW6 1XJ


## 4. Build canonical address keys (Methods A–F)

Each method encodes a different combination of OS address components into a single
string. Using multiple encodings increases the chance that at least one variant
matches the LDC input:

- **A** — org name + street + full postcode *(best for named organisations)*
- **B** — sub-building + number + street + full postcode
- **C** — building name + number + street + full postcode
- **D** — building name + sub-building + street + full postcode
- **E** — number + street + full postcode *(most minimal, highest recall)*
- **F** — building name + sub-building + number + street + full postcode

In [52]:
canonical_a = build_canonical_key(os_df, ["organisationname", "streetname", "postcode"], "ca")
canonical_b = build_canonical_key(os_df, ["subname",          "number_str",  "streetname", "postcode"], "cb")
canonical_c = build_canonical_key(os_df, ["name",             "number_str",  "streetname", "postcode"], "cc")
canonical_d = build_canonical_key(os_df, ["name",             "subname",     "streetname", "postcode"], "cd")
canonical_e = build_canonical_key(os_df, ["number_str",       "streetname",  "postcode"],               "ce")
canonical_f = build_canonical_key(os_df, ["name",     "subname", "number_str", "streetname", "postcode"], "cf")

for label, df in [("A", canonical_a), ("B", canonical_b), ("C", canonical_c),
                  ("D", canonical_d), ("E", canonical_e), ("F", canonical_f)]:
    print(f"Canonical {label}: {len(df):,} rows — e.g. {df['address_c'].iloc[0]!r}")

Canonical A: 1,340,432 rows — e.g. 'BARLOW ROAD, NW6 2BH'
Canonical B: 1,315,815 rows — e.g. 'UNIT 5, BARLOW ROAD, NW6 2BH'
Canonical C: 1,299,001 rows — e.g. 'LAURISTON LODGE, BARLOW ROAD, NW6 2BH'
Canonical D: 1,353,249 rows — e.g. 'LAURISTON LODGE, FLAT 5, BARLOW ROAD, NW6 2BH'
Canonical E: 1,261,797 rows — e.g. 'BARLOW ROAD, NW6 2BH'
Canonical F: 1,342,201 rows — e.g. 'LAURISTON LODGE, FLAT 5, BARLOW ROAD, NW6 2BH'


## 5. Load LDC messy data

The LDC CSV has a single concatenated `address` field and an `organisationname`
(`tenant`). We split the address on `, ` (right-aligning so postcode is always
the last token), then drop the post town and city fields that OS doesn't match on.

In [53]:
ldc_raw = pd.read_csv(LDC_FILE)
ldc_raw["address"] = (
    ldc_raw["address"]
    .str.replace("'", "", regex=False)
    .str.replace(".", "", regex=False)
)
print(f"LDC rows: {len(ldc_raw):,}")
ldc_raw[["premises_id", "tenant", "address", "uprn_id"]].head(3)

LDC rows: 347,378


,premises_id,tenant,address,uprn_id
0,52054865,Vacant Property,"125-125A, Northcote Road, London, Greater Lond...",NaN
1,52679951,Vacant Property,"Cabot Square, London, Greater London, E14 4QQ",NaN
2,53132466,Ildy Gift Shop,"Unit 8, Pavilions Shopping Centre, Pantile Wal...",NaN


In [54]:
# Split the concatenated address into columns, right-shift so postcode is last
ldc_split = ldc_raw["address"].str.split(", ", n=10, expand=True)
for _ in range(ldc_split.shape[1]):
    no_postcode = ldc_split.index[ldc_split[ldc_split.shape[1] - 1].isna()].tolist()
    ldc_split.iloc[no_postcode] = ldc_split.iloc[no_postcode].shift(periods=1, axis=1)

# Drop post-town (col -2) and city (col -3), add org name, UPRN, sector
n = ldc_split.shape[1]
ldc_df = ldc_split.drop(columns=[n - 3, n - 2])
ldc_df["organisationname"] = ldc_raw["tenant"]
ldc_df["uprn"]             = ldc_raw["uprn_id"]
ldc_df["premises_id"]      = ldc_raw["premises_id"]
ldc_df["postcode"]         = ldc_df[n - 1]
ldc_df["postcode_sector"]  = ldc_df["postcode"].str[:-2]
ldc_df["streetname"]       = ldc_df[n - 4]
ldc_df = ldc_df.drop_duplicates()

print(f"LDC after splitting: {len(ldc_df):,} rows")
ldc_df.head(2)

LDC after splitting: 280,423 rows


,0,1,2,3,4,7,organisationname,uprn,premises_id,postcode,postcode_sector,streetname
0,None,None,None,125-125A,Northcote Road,SW11 6PS,Vacant Property,NaN,52054865,SW11 6PS,SW11 6,Northcote Road
1,None,None,None,None,Cabot Square,E14 4QQ,Vacant Property,NaN,52679951,E14 4QQ,E14 4,Cabot Square


## 6. Build messy address keys (Methods a–b)

The LDC data doesn't have distinct OS-style fields (building name, sub-building etc.),
so we only need two messy variants:

- **a** — org name + street + full postcode *(matches canonical A)*
- **b** — full raw address tokens + full postcode *(matches canonicals B–F)*

In [55]:
# Identify the token columns (everything between col 0 and streetname/postcode)
_addr_token_cols = [c for c in ldc_df.columns if isinstance(c, int) and c <= n - 5]

messy_a = build_messy_key(ldc_df, ["organisationname", "streetname", "postcode"], "ma")
messy_b = build_messy_key(ldc_df, _addr_token_cols + ["streetname", "postcode"], "mb")

print(f"Messy A: {len(messy_a):,} rows — e.g. {messy_a['address_m'].iloc[0]!r}")
print(f"Messy B: {len(messy_b):,} rows — e.g. {messy_b['address_m'].iloc[0]!r}")

Messy A: 280,329 rows — e.g. 'VACANT PROPERTY, WESTBOURNE PARK ROAD, W11 1EH'
Messy B: 161,796 rows — e.g. '284, WESTBOURNE PARK ROAD, W11 1EH'


## 7. Run 6 matching rounds

For each (messy, canonical) pair we:
1. Register the DataFrames as DuckDB relations
2. Run `AddressMatcher` (exact pass → Splink probabilistic pass)
3. Collect the raw match result
4. Join back to the original messy/canonical frames to recover `address_m`,
   `address_c`, `premises_id`, and `uprn`

The `final_match_weight_threshold=8` means Splink will only return pairs with
match weight ≥ 8 (~94% probability). Lower this value to trade precision for
recall.

In [41]:
import tempfile as _tmpmod
_TMP_MESSY_PQ  = os.path.join(_tmpmod.gettempdir(), "_ukam_messy.parquet")
_TMP_CANON_PQ  = os.path.join(_tmpmod.gettempdir(), "_ukam_canon.parquet")

pair_list = [
    (messy_a, canonical_a, "a"),
    (messy_b, canonical_b, "b"),
    (messy_b, canonical_c, "c"),
    (messy_b, canonical_d, "d"),
    (messy_b, canonical_e, "e"),
    (messy_b, canonical_f, "f"),
]

all_results = []
t0 = time.time()

for i, (df_messy_pd, df_canon_pd, method) in enumerate(pair_list):
    uid_m_col = [c for c in df_messy_pd.columns if c.startswith("uid_")][0]
    uid_c_col = [c for c in df_canon_pd.columns if c.startswith("uid_")][0]

    # Prepare input/ref DataFrames with the columns AddressMatcher expects
    # Include an explicit 'postcode' column so the library doesn't rely on
    # regex extraction from address_concat (sector codes would fail the regex).
    input_df = df_messy_pd.rename(columns={uid_m_col: "unique_id", "address_m": "address_concat"})[["unique_id", "address_concat"]].copy()
    input_df["postcode"] = input_df["address_concat"].str.rsplit(", ", n=1).str[-1]

    ref_df = df_canon_pd.rename(columns={uid_c_col: "unique_id", "address_c": "address_concat"})[["unique_id", "address_concat"]].copy()
    ref_df["postcode"] = ref_df["address_concat"].str.rsplit(", ", n=1).str[-1]

    # Save to parquet and load via a FRESH DuckDB connection per round
    input_df.to_parquet(_TMP_MESSY_PQ)
    ref_df.to_parquet(_TMP_CANON_PQ)

    round_con = duckdb.connect(database=":memory:")
    round_con.execute(f"PRAGMA threads={cores}")
    if container_memory:
        round_con.execute(f"PRAGMA memory_limit='{container_memory}'")
    else:
        round_con.execute(f"PRAGMA memory_limit='{mem_limit}GB'")
    round_con.execute(f"SET temp_directory='{temp_dir}'")

    pq_input = round_con.read_parquet(_TMP_MESSY_PQ)
    pq_ref   = round_con.read_parquet(_TMP_CANON_PQ)

    print(f"Round {i} (method '{method}') — {df_messy_pd.shape[0]:,} messy x {df_canon_pd.shape[0]:,} canonical")
    t_round = time.time()

    matcher = AddressMatcher(
        canonical_addresses=pq_ref,
        addresses_to_match=pq_input,
        con=round_con,
        stages=[
            ExactMatchStage(),
            PeeledAddressStage(),
            UniqueTrigramStage(),
            SplinkStage(
                predict_threshold_match_weight=-20,
                final_match_weight_threshold=8,
                include_full_postcode_block=False,
                retain_intermediate_calculation_columns=False,
            ),
        ],
    )

    match_result = matcher.match()

    # Extract results — filter to matched rows only
    df_result = match_result.matches().df()
    df_result = df_result[df_result["resolved_canonical_id"].notna()]
    df_result = df_result.rename(columns={
        "unique_id": uid_m_col,
        "resolved_canonical_id": uid_c_col,
        "original_address_concat": "address_m",
        "original_address_concat_canonical": "address_c",
    })

    if df_result.empty:
        print(f"  No matches found — skipping.")
        round_con.close()
        continue

    df_result["fuzz_similarity"] = df_result.apply(fuzzy_similarity, axis=1)

    # Join back to original messy/canonical to recover premises_id, uprn
    combined_df = df_messy_pd.merge(
        df_result[[uid_m_col, uid_c_col, "match_weight", "distinguishability", "fuzz_similarity"]],
        on=uid_m_col,
        how="left",
    ).merge(
        df_canon_pd,
        on=uid_c_col,
        how="left",
        suffixes=["_m", "_c"],
    ).sort_values(
        ["premises_id", "fuzz_similarity"],
        ascending=[True, False],
    )
    combined_df["method"] = method

    all_results.append(combined_df)
    matched_count = combined_df[uid_c_col].notna().sum()
    round_con.close()
    print(f"  -> {matched_count:,} matched / {len(combined_df):,} total in {time.time() - t_round:.1f}s")

print(f"\nAll rounds done in {time.time() - t0:.1f}s")

Round 0 (method 'a') — 280,329 messy x 1,340,432 canonical
  -> 110,658 matched / 280,329 total in 69.2s
Round 1 (method 'b') — 161,796 messy x 1,315,815 canonical
  -> 142,487 matched / 161,796 total in 65.2s
Round 2 (method 'c') — 161,796 messy x 1,299,001 canonical
  -> 139,649 matched / 161,796 total in 70.8s
Round 3 (method 'd') — 161,796 messy x 1,353,249 canonical
  -> 56,587 matched / 161,796 total in 82.8s
Round 4 (method 'e') — 161,796 messy x 1,261,797 canonical
  -> 137,442 matched / 161,796 total in 54.5s
Round 5 (method 'f') — 161,796 messy x 1,342,201 canonical
  -> 144,448 matched / 161,796 total in 83.8s

All rounds done in 462.0s


## 8. Combine results and compute fuzzy similarity

`fuzz_similarity` is the Levenshtein ratio between the LDC string we sent into
the matcher (`address_m`) and the OS string it matched against (`address_c`).
A score of 100 means the strings were identical; ≥ 95 is near-identical.

In [42]:
full_results = pd.concat(all_results, ignore_index=True)

# Normalise column names — handle both 'uprn' and 'uprn_c'/'uprn_m'
if "uprn" in full_results.columns and "uprn_c" not in full_results.columns:
    full_results = full_results.rename(columns={"uprn": "uprn_c"})

# Ensure consistent types
full_results["premises_id"] = full_results["premises_id"].astype(str)
if "uprn_c" in full_results.columns:
    full_results["uprn_c"] = full_results["uprn_c"].astype(str).str.replace(r'\.0$', '', regex=True)

matched = full_results[full_results["match_weight"].notna()]

print(f"Total rows         : {len(full_results):,}")
print(f"Unique premises    : {full_results['premises_id'].nunique():,}")
print(f"Premises with match: {matched['premises_id'].nunique():,}")
print(f"\nfuzz_similarity distribution (matched rows only):")
print(matched["fuzz_similarity"].describe().to_string())

Total rows         : 1,089,309
Unique premises    : 127,161
Premises with match: 55,616

fuzz_similarity distribution (matched rows only):
count    144308.000000
mean         84.163927
std          12.147659
min          29.000000
25%          75.000000
50%          88.000000
75%          95.000000
max          99.000000


In [44]:
# Export for use in labelling_accuracy.ipynb
full_results.to_csv("full_results_new.csv", index=False)
print(f"Exported {len(full_results):,} rows to full_results_new.csv")

Exported 1,089,309 rows to full_results_new.csv


## 8b. Compare against colleague's results (optional)

If `results.csv` exists, this section loads it and compares it against the
`full_results` you just produced from the pipeline above.

The comparison answers:
- **Coverage** — did the new run match more or fewer premises overall?
- **Agreement** — where both runs matched the same `premises_id`, do they agree on the UPRN?
- **Gains** — premises newly matched in your run (not in colleague's)
- **Losses** — premises in colleague's run that your run missed
- **Conflicts** — same premise, different UPRN (worth manually inspecting)

In [43]:
if not os.path.exists(RESULTS_CSV):
    print(f"No reference file found at '{RESULTS_CSV}' — skipping comparison.")
else:
    ref = pd.read_csv(RESULTS_CSV, dtype={"premises_id": str, "uprn_c": str})
    ref["premises_id"] = ref["premises_id"].astype(str)
    ref["uprn_c"] = ref["uprn_c"].astype(str).str.replace(r'\.0$', '', regex=True)

    def hit_premises_from(df: pd.DataFrame) -> dict:
        """Apply p1/p2/p3 rules and return {premises_id: uprn_c} for accepted hits."""
        def best(subset):
            if subset.empty:
                return pd.DataFrame(columns=["premises_id", "uprn_c"])
            return (
                subset.sort_values("match_weight", ascending=False)
                .groupby("premises_id")
                .first()
                .reset_index()[["premises_id", "uprn_c"]]
            )
        r1 = best(df[df["fuzz_similarity"] == 100])
        r2 = best(df[(df["fuzz_similarity"] > 95) & (df["method"] == "a")])
        r3 = best(df[df["distinguishability"].notna()])
        combined = pd.concat([r1, r2, r3]).drop_duplicates(subset="premises_id", keep="first")
        return dict(zip(combined["premises_id"], combined["uprn_c"]))

    if ref["fuzz_similarity"].fillna(0).eq(0).all():
        ref["fuzz_similarity"] = ref.apply(fuzzy_similarity, axis=1)

    new_hits = hit_premises_from(full_results)
    ref_hits = hit_premises_from(ref)

    new_set = set(new_hits)
    ref_set = set(ref_hits)
    both    = new_set & ref_set
    gains   = new_set - ref_set
    losses  = ref_set - new_set

    agree    = sum(new_hits[p] == ref_hits[p] for p in both)
    conflict = [(p, new_hits[p], ref_hits[p]) for p in both if new_hits[p] != ref_hits[p]]

    all_premises_new = set(full_results["premises_id"].dropna())
    all_premises_ref = set(ref["premises_id"].dropna())

    print("━" * 55)
    print(f"{'':30s} {'New run':>10s}  {'Colleague':>10s}")
    print("━" * 55)
    print(f"{'Total premises seen':30s} {len(all_premises_new):>10,}  {len(all_premises_ref):>10,}")
    print(f"{'Hit premises (any rule)':30s} {len(new_set):>10,}  {len(ref_set):>10,}")
    cov_new = len(new_set) / len(all_premises_new) if all_premises_new else 0
    cov_ref = len(ref_set) / len(all_premises_ref) if all_premises_ref else 0
    print(f"{'Coverage':30s} {cov_new:>10.1%}  {cov_ref:>10.1%}")
    print("━" * 55)
    print(f"{'Premises matched by BOTH':30s} {len(both):>10,}")
    print(f"  → agree on UPRN         {agree:>10,}")
    print(f"  → conflict (diff UPRN)  {len(conflict):>10,}")
    print(f"{'Gains  (new only)':30s} {len(gains):>10,}")
    print(f"{'Losses (colleague only)':30s} {len(losses):>10,}")
    print("━" * 55)

    if conflict:
        print(f"\nFirst {min(10, len(conflict))} UPRN conflicts:")
        print(f"  {'premises_id':>12s}  {'new UPRN':>14s}  {'colleague UPRN':>14s}  address")
        for pid, n_uprn, r_uprn in conflict[:10]:
            addr = full_results.loc[full_results["premises_id"] == pid, "address_m"]
            addr_str = addr.iloc[0][:50] if len(addr) else ""
            print(f"  {pid:>12s}  {str(n_uprn):>14s}  {str(r_uprn):>14s}  {addr_str}")

    if gains:
        print(f"\nSample gains (premises your run matched; colleague's did not):")
        for pid in list(gains)[:5]:
            addr = full_results.loc[full_results["premises_id"] == pid, "address_m"].iloc[0]
            print(f"  {pid:>12s}  UPRN {new_hits[pid]}  {addr[:50]}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                                  New run   Colleague
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total premises seen               127,161     127,161
Hit premises (any rule)           115,191      97,883
Coverage                            90.6%       77.0%
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Premises matched by BOTH           95,971
  → agree on UPRN             90,215
  → conflict (diff UPRN)       5,756
Gains  (new only)                  19,220
Losses (colleague only)             1,912
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

First 10 UPRN conflicts:
   premises_id        new UPRN  colleague UPRN  address
      50324111         5204763         5087071  MAIL BOXES ETC, HIGHGATE HIGH STREET, N6 5JT
      50481209     10008332424     10008297888  MANOR KEBAB, SEVEN SISTERS ROAD, N4 2DE
      50007427    100023431225     10033583681  L'ETO, LONG ACRE, WC2E 9AD
      50021694        

## 9. Apply hit rules

A **hit** is a match we are confident enough to accept.  Three independent rules
each contribute a set of `premises_id` values:

| Rule | Condition | Rationale |
|------|-----------|-----------|
| **p1** | `fuzz_similarity == 100` | Exact string match — zero ambiguity |
| **p2** | `fuzz_similarity > 95` **and** `method == 'a'` | Org-name round, near-exact |
| **p3** | `distinguishability IS NOT NULL` | Splink found a unique token |

For each rule, when a `premises_id` appears in multiple rows (different methods
or match rounds) we take the **highest `match_weight`** row as the accepted match.

In [ ]:
def top_by_weight(df: pd.DataFrame) -> pd.Series:
    """Return series of premises_id values that have at least one qualifying row."""
    return (
        df.sort_values("match_weight", ascending=False)
        .groupby("premises_id")
        .first()
        .reset_index()["premises_id"]
    )


p1 = top_by_weight(full_results[full_results["fuzz_similarity"] == 100])
p2 = top_by_weight(full_results[(full_results["fuzz_similarity"] > 95) & (full_results["method"] == "a")])
p3 = top_by_weight(full_results[full_results["distinguishability"].notna()])

hit_premises = set(p1) | set(p2) | set(p3)
all_premises = set(full_results["premises_id"].dropna())

coverage = len(hit_premises) / len(all_premises) if all_premises else 0

print(f"Rule p1 (exact string match)     : {len(p1):,} premises")
print(f"Rule p2 (org near-exact, fuzz>95): {len(p2):,} premises")
print(f"Rule p3 (distinguishability)     : {len(p3):,} premises")
print(f"Union (any rule)                 : {len(hit_premises):,} / {len(all_premises):,}")
print(f"\nCoverage: {coverage:.1%}")

## 10. Explore results

The cells below let you inspect which premises were and were not matched,
and dig into edge cases.

In [ ]:
# Premises NOT matched by any rule — review these to find new hit criteria
unmatched = full_results[~full_results["premises_id"].isin(hit_premises)]
print(f"Unmatched premises: {unmatched['premises_id'].nunique():,}")
unmatched[["premises_id", "address_m", "address_c", "match_weight", "fuzz_similarity", "method"]].head(10)

In [ ]:
# Coverage breakdown by method
full_results.groupby("method").agg(
    rows=("premises_id", "count"),
    premises=("premises_id", "nunique"),
    avg_fuzz=("fuzz_similarity", "mean"),
    avg_weight=("match_weight", "mean"),
).round(1)

In [ ]:
# Best accepted match for each premises (one row per premises_id)
best_matches = (
    full_results[full_results["premises_id"].isin(hit_premises)]
    .sort_values("match_weight", ascending=False)
    .groupby("premises_id")
    .first()
    .reset_index()
)
print(f"Best-match rows: {len(best_matches):,}")
best_matches[["premises_id", "uprn_c", "address_m", "address_c",
              "match_weight", "fuzz_similarity", "method"]].head(10)

## 11. Export 100 rows for manual labelling

To calibrate the hit rules (and find a good `final_match_weight_threshold`
for the Splink stage) we need ground-truth labels from a human reviewer.

This cell exports a **stratified random sample of 100 rows** — spread across
low / medium / high match-weight bands — to a CSV that you can open in Excel
or Google Sheets.  For each row, fill in:

| Column | What to fill in |
|--------|----------------|
| `human_label` | **1** if the canonical address (`address_c`) is the correct match for the messy address (`address_m`), **0** if it is wrong or uncertain |
| `true_uprn` | If `human_label=0` **and** you know the correct UPRN, write it here; otherwise leave blank |

Save the filled file back as `labels_100.csv` in the same folder, then run
Section 12 to get precision/recall curves and threshold recommendations.

In [ ]:
import numpy as np

LABELS_FILE = os.path.join(DATA_DIR, "ldc", "labels_100.csv")  # output path — same folder as LDC_FILE
N_SAMPLE    = 100
RANDOM_SEED = 42

# Work from full_results (all rounds, before hit-rule filtering) so we cover
# both confident and borderline rows.
pool = full_results.copy()

# Deduplicate: keep the best row per premises_id (highest match_weight)
pool = (
    pool.sort_values("match_weight", ascending=False)
    .groupby("premises_id")
    .first()
    .reset_index()
)

# Stratify into three weight bands
low    = pool[pool["match_weight"] <  -2]
medium = pool[(pool["match_weight"] >= -2) & (pool["match_weight"] < 5)]
high   = pool[pool["match_weight"] >=  5]

n_low  = min(20, len(low))
n_mid  = min(40, len(medium))
n_high = min(40, len(high))

# If any band is under-represented, fill remaining slots from the largest band
n_total = n_low + n_mid + n_high
shortfall = N_SAMPLE - n_total
if shortfall > 0:
    largest = max([(len(low), "low"), (len(medium), "medium"), (len(high), "high")], key=lambda x: x[0])[1]
    if largest == "high":    n_high += shortfall
    elif largest == "medium": n_mid  += shortfall
    else:                     n_low  += shortfall

rng = np.random.default_rng(RANDOM_SEED)

sample = pd.concat([
    low.sample(n=n_low,  random_state=RANDOM_SEED) if n_low  > 0 else pd.DataFrame(),
    medium.sample(n=n_mid,  random_state=RANDOM_SEED) if n_mid  > 0 else pd.DataFrame(),
    high.sample(n=n_high, random_state=RANDOM_SEED) if n_high > 0 else pd.DataFrame(),
], ignore_index=True)

export_cols = [
    "premises_id",     # messy unique_id
    "uprn_c",          # candidate canonical unique_id (UPRN)
    "address_m",       # messy address text
    "address_c",       # canonical address text
    "postcode_m",      # messy postcode
    "postcode_c",      # canonical postcode
    "match_weight",
    "fuzz_similarity",
    "method",
    "distinguishability",
]
# Keep only columns that actually exist in the dataframe
export_cols = [c for c in export_cols if c in sample.columns]

label_df = sample[export_cols].copy()
label_df["human_label"] = ""   # reviewer fills 1 (hit) or 0 (miss)
label_df["true_uprn"]   = ""   # reviewer fills correct UPRN if human_label=0

os.makedirs(os.path.dirname(LABELS_FILE), exist_ok=True)
label_df.to_csv(LABELS_FILE, index=False)

print(f"Exported {len(label_df)} rows to: {LABELS_FILE}")
print(f"  Low  weight (<-2)  : {n_low}")
print(f"  Med  weight (-2..5): {n_mid}")
print(f"  High weight (>=5)  : {n_high}")
print()
print("Open the CSV, fill 'human_label' (1=hit, 0=miss) and 'true_uprn' where known,")
print("save as labels_100.csv, then run Section 12.")
label_df.head()

## 12. Accuracy analysis and threshold recommendation

Once `labels_100.csv` is filled in, this section:

1. Reads the labels and builds a messy table with a `ukam_label` column
   (the ground-truth UPRN for confirmed hits).
2. Runs the Splink matching stage on the labelled rows.
3. Calls `MatchResult.accuracy_analysis()` to sweep over all possible
   `final_match_weight_threshold` values and plot precision vs recall.
4. Prints the threshold table so you can choose a value and paste it
   into `SplinkStage(final_match_weight_threshold=...)` above.

> **Why re-run matching here?**  
> `accuracy_analysis()` needs a `MatchResult` object that has access to
> the raw Splink scores *before* the threshold is applied.  We therefore
> run a fresh matching pass on just the 100 labelled rows.

In [ ]:
from uk_address_matcher import AddressMatcher, ExactMatchStage, SplinkStage

# ── 12a. Load the filled-in labels CSV ─────────────────────────────────────
labels_df = pd.read_csv(LABELS_FILE)

# Keep only rows where the reviewer provided a judgment
labels_df = labels_df[labels_df["human_label"].notna() & (labels_df["human_label"] != "")]
labels_df["human_label"] = labels_df["human_label"].astype(int)

print(f"Labelled rows loaded : {len(labels_df)}")
print(f"  Hits  (label=1)    : {(labels_df['human_label']==1).sum()}")
print(f"  Misses (label=0)   : {(labels_df['human_label']==0).sum()}")

In [ ]:
# ── 12b. Build messy table with ukam_label ──────────────────────────────────
# ukam_label = confirmed UPRN for hits; true_uprn (if known) for misses;
# NULL otherwise (row is treated as a "negative" in the evaluation).

def resolve_ukam_label(row):
    if row["human_label"] == 1:
        return str(row["uprn_c"])  # confirmed correct match
    true = row.get("true_uprn", "")
    if pd.notna(true) and str(true).strip():
        return str(true)           # reviewer supplied the correct UPRN
    return None                    # unknown — exclude from positives

labels_df["ukam_label"] = labels_df.apply(resolve_ukam_label, axis=1)

# The messy table needs: unique_id, address_concat, postcode, ukam_label
address_col = "address_m" if "address_m" in labels_df.columns else "address_concat"
postcode_col = "postcode_m" if "postcode_m" in labels_df.columns else "postcode"

messy_labels = labels_df[["premises_id", address_col, postcode_col, "ukam_label"]].rename(
    columns={"premises_id": "unique_id", address_col: "address_concat", postcode_col: "postcode"}
)

# Register as a DuckDB relation
con.execute("CREATE OR REPLACE TABLE __label_messy__ AS SELECT * FROM messy_labels")
label_messy_rel = con.table("__label_messy__")

print("Messy label table preview:")
print(messy_labels.head())

In [ ]:
# ── 12c. Re-run matching on the labelled rows ───────────────────────────────
# We need a canonical table.  If the pipeline was run above, canonical_raw
# already exists in DuckDB.  Otherwise we build a minimal version from the
# uprn_c / address_c columns in the pre-computed results.

try:
    con.table("canonical_raw")
    print("Using canonical_raw from the pipeline run.")
except Exception:
    print("canonical_raw not found — building from results.csv uprn_c/address_c columns...")
    if "full_results" not in dir() or full_results is None:
        raise RuntimeError("Run Section 2b first to load full_results from results.csv.")
    canonical_from_results = (
        full_results[["uprn_c", "address_c"]]
        .dropna(subset=["uprn_c", "address_c"])
        .drop_duplicates(subset=["uprn_c"])
        .rename(columns={"uprn_c": "unique_id", "address_c": "address_concat"})
        .assign(postcode=lambda d: d["address_concat"].str.extract(r'([A-Z]{1,2}\d[\d\w]? \d[A-Z]{2})')[0])
    )
    con.execute("CREATE OR REPLACE TABLE canonical_raw AS SELECT * FROM canonical_from_results")
    print(f"Built canonical_raw: {len(canonical_from_results):,} unique addresses.")

label_matcher = AddressMatcher(
    canonical_addresses=con.table("canonical_raw"),
    addresses_to_match=label_messy_rel,
    con=con,
    stages=[
        ExactMatchStage(),
        SplinkStage(
            predict_threshold_match_weight=-20,  # low floor — keep everything
            final_match_weight_threshold=None,   # no cut — sweep in accuracy_analysis
        ),
    ],
)
label_result = label_matcher.match()
print("Matching complete — running accuracy analysis...")

In [ ]:
# ── 12d. Threshold sweep table ──────────────────────────────────────────────
# Shows precision & recall at every possible match_weight threshold.
# Pick the row that gives you the precision/recall balance you want,
# then use that truth_threshold as final_match_weight_threshold in SplinkStage.

threshold_table = label_result.accuracy_analysis(output_type="table")
threshold_df = pd.DataFrame(threshold_table)

# Filter to the most useful range and round for readability
display_df = (
    threshold_df[
        (threshold_df["truth_threshold"].abs() < 900)  # drop sentinel rows
    ][["truth_threshold", "precision", "recall", "f1", "tp", "fp", "fn"]]
    .sort_values("truth_threshold")
    .round({"truth_threshold": 1, "precision": 3, "recall": 3, "f1": 3})
    .reset_index(drop=True)
)

print("Precision / Recall sweep (choose a truth_threshold for SplinkStage):")
print(display_df.to_string(index=False))

In [ ]:
# ── 12e. Interactive chart (requires altair) ────────────────────────────────
# Uncomment if altair is installed:  uv add altair --group notebooks

# chart = label_result.accuracy_analysis(output_type="threshold_selection")
# chart  # displays in Jupyter

# Or for a precision-recall curve:
# pr_chart = label_result.accuracy_analysis(output_type="precision_recall")
# pr_chart

# Matplotlib fallback — always works ────────────────────────────────────────
import matplotlib.pyplot as plt

plot_df = display_df.dropna(subset=["precision", "recall"])
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(plot_df["recall"], plot_df["precision"], marker="o", linewidth=2)

# Annotate every other point with the threshold value
for _, row in plot_df.iloc[::2].iterrows():
    ax.annotate(
        f"{row['truth_threshold']:.1f}",
        xy=(row["recall"], row["precision"]),
        xytext=(4, 4),
        textcoords="offset points",
        fontsize=8,
    )

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision–Recall curve (labels = 100 manually reviewed rows)\n"
             "Annotated numbers = match_weight threshold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ─── Recommendation ─────────────────────────────────────────────────────────
best = display_df.loc[display_df["f1"].idxmax()]
print(f"\nHighest F1 = {best['f1']:.3f} at match_weight threshold = {best['truth_threshold']:.1f}")
print(f"  → set SplinkStage(final_match_weight_threshold={best['truth_threshold']:.1f}) in Section 6")